### Recuperação das credenciais do SQL Server

Nesta etapa são recuperadas as credenciais armazenadas no Databricks Secret Scope.

A utilização do Secret Scope permite acessar informações sensíveis, como servidor, banco de dados, usuário e senha, sem expor esses dados diretamente no código do notebook, aumentando a segurança da solução.

In [0]:
jdbc_hostname = dbutils.secrets.get(
    scope="mercadata-sqlserver",
    key="jdbc_hostname"
)

jdbc_database = dbutils.secrets.get(
    scope="mercadata-sqlserver",
    key="jdbc_database"
)

jdbc_username = dbutils.secrets.get(
    scope="mercadata-sqlserver",
    key="jdbc_username"
)

jdbc_password = dbutils.secrets.get(
    scope="mercadata-sqlserver",
    key="jdbc_password"
)

jdbc_url = (
    f"jdbc:sqlserver://{jdbc_hostname};"
    f"databaseName={jdbc_database}"
)

connection_properties = {
    "user": jdbc_username,
    "password": jdbc_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

df_pedidos = spark.read.jdbc(
    url=jdbc_url,
    table="ecommerce_pedidos",
    properties=connection_properties
)

print("Tabela carregada com sucesso!")

### Camada Bronze

A camada Bronze representa o primeiro estágio da arquitetura medalhão. Seu objetivo é armazenar os dados provenientes da origem com o menor número possível de transformações, preservando sua estrutura original.

Nesta etapa, os dados da tabela `ecommerce_pedidos`, provenientes do SQL Server, serão carregados para uma tabela Delta no Unity Catalog utilizando o modo de escrita `append`.

A utilização do formato Delta Lake permite controle transacional, versionamento dos dados e maior eficiência em operações futuras de processamento.


In [0]:
spark.sql("SHOW SCHEMAS IN workspace").show(truncate=False)

#Análise dos catologos criados.

In [0]:
spark.sql("SHOW SCHEMAS IN workspace").show(truncate=False)

#Análise dos Schemas criados.

###Estrutura da Arquitetura Medalhão

Para organizar os dados de acordo com a arquitetura medalhão, serão criados três schemas no Unity Catalog:

Bronze: armazenamento dos dados brutos provenientes da origem.
Silver: armazenamento dos dados tratados e padronizados.
Gold: armazenamento dos dados agregados e preparados para consumo analítico.

Essa separação facilita a governança, rastreabilidade e manutenção dos pipelines de dados.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

#Criação dos Schemas para as 3 camadas, para facilitar a organização.

In [0]:
spark.sql("SHOW SCHEMAS IN workspace").show(truncate=False)

#Validação do schemas das camadas criados.

### Adição de metadados de carga

Nesta etapa é adicionada a coluna data_carga ao DataFrame da camada Bronze.

Essa coluna registra o momento em que os dados foram ingeridos na plataforma, permitindo rastrear quando cada carga foi realizada e facilitando auditorias, monitoramento e análises históricas do processo de ingestão.

A inclusão de metadados de carga é uma prática comum em arquiteturas de dados modernas, contribuindo para a governança e rastreabilidade das informações.

In [0]:
#Primeiro vamos adicionar uma coluna técnica de carga.

from pyspark.sql.functions import current_timestamp

df_bronze = (
    df_pedidos
    .withColumn("data_carga", current_timestamp())
)



In [0]:
df_bronze.printSchema()

#Conferindo se a tabela está com a coluna data_carga.

### Persistência da camada Bronze

Após a inclusão dos metadados de carga, os dados serão persistidos na camada Bronze utilizando o formato Delta Lake.

A gravação será realizada no modo append, permitindo que novas cargas sejam adicionadas à tabela sem sobrescrever os dados já existentes.

O uso do Delta Lake oferece recursos como transações ACID, versionamento, controle de schema e melhor desempenho para processamento analítico.

In [0]:
df_bronze.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.bronze.ecommerce_pedidos")

### Validação da camada Bronze

Após a persistência dos dados na camada Bronze, foi realizada uma validação para confirmar a criação da tabela Delta e verificar a disponibilidade dos registros armazenados.

Essa etapa tem como objetivo garantir que os dados provenientes do SQL Server foram carregados corretamente para o Unity Catalog, preservando sua estrutura original e os metadados de ingestão adicionados durante o processo.

A validação consiste na consulta dos registros armazenados na tabela workspace.bronze.ecommerce_pedidos, permitindo verificar a integridade dos dados e confirmar o sucesso da carga realizada na camada Bronze

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM workspace.bronze.ecommerce_pedidos
""").show()

In [0]:
display(
    spark.table("workspace.bronze.ecommerce_pedidos")
)